# Landing → Bronze — `<TABELA>` (`<Responsável>`)

**Antes de preencher:** troque `<TABELA>` e `<Responsável>` no título acima e no nome do arquivo quando
salvar a sua cópia (ex.: `Landing_To_Bronze_custos_milton.ipynb`). Não edite o template original — faça
uma cópia por tabela.

Regras da camada bronze (ver `CLAUDE.md`):
- **Fiel à origem**: tudo lido como `string`, nenhuma correção de conteúdo aqui (isso é papel da silver).
- **Idempotente**: escreve com `overwrite` — rodar de novo sobre o mesmo CSV dá o mesmo resultado.
- **Rastreável**: toda tabela ganha `_ingestion_ts` (quando foi ingerida) e `_source_file` (de qual
  arquivo veio).
- **Sem dado pessoal em output**: se a sua tabela for `clientes`, nunca dê `display()`/`show()` em linhas
  — só em contagens/agregados.

Tabelas e responsáveis (ver `analises/README.md`): Yuri → `clientes` · Cris → `base_atendentes`,
`base_motivos` · Juan → `chamados_hora` · Richard → `canais` · Milton → `custos`, `pesquisa_satisfacao`.
Todo mundo também pode precisar de `chamados`.

## 1. Configuração — preencha os `TODO`

In [ ]:
dbutils.widgets.text("base_path", "")
BASE_PATH = dbutils.widgets.get("base_path")
if not BASE_PATH:
    raise ValueError(
        "Preencha o widget 'base_path' com um caminho de Volume: '/Volumes/<catalogo>/<schema>/<volume>'."
    )
if not BASE_PATH.startswith("/Volumes"):
    raise ValueError(
        f"'{BASE_PATH}' não é um Volume. O Spark serverless não lê CSV de pasta do Workspace — "
        "suba os CSVs num Volume do Unity Catalog e use '/Volumes/<catalogo>/<schema>/<volume>'."
    )

# TODO: nome exato do arquivo, igual está em data/raw/ (cuidado com maiúscula/minúscula)
# clientes.csv · Chamados_Hora.CSV · custos.csv · pesquisa_satisfacao.csv
# base_atendentes.csv · base_motivos.csv · canais.csv · chamados.csv
ARQUIVO = "<NOME_DO_ARQUIVO.csv>"

# TODO: encoding do arquivo — "ISO-8859-1" só para clientes.csv e Chamados_Hora.CSV; "UTF-8" para os demais
ENCODING = "UTF-8"

# TODO: nome da tabela de destino na bronze (minúsculo, sem espaço — ex.: "clientes", "custos", "chamados_hora")
TABELA = "<nome_tabela>"

CATALOGO = "workspace"
SCHEMA_BRONZE = "vcredit_bronze"

## 2. Leitura (fiel à origem — tudo `string`)

`inferSchema=False` sempre: deixar o Spark adivinhar o tipo é arriscado aqui (ex.: `id_cliente` é CPF
com zero à esquerda — se virar número, perde o zero e corrompe o dado). `Chamados_Hora.CSV` é a única
fonte sem aspas; as demais usam aspas duplas.

In [ ]:
from pyspark.sql import functions as F

leitor = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("encoding", ENCODING)
)
if ARQUIVO != "Chamados_Hora.CSV":
    leitor = leitor.option("quote", '"').option("escape", '"')

df = leitor.csv(f"{BASE_PATH}/{ARQUIVO}")

# padroniza nomes de coluna (remove BOM/espaços à toa, minúsculo) — não muda o conteúdo das células
for antigo in df.columns:
    novo = antigo.replace("\ufeff", "").strip().lower()
    if novo != antigo:
        df = df.withColumnRenamed(antigo, novo)

print(f"{ARQUIVO}: {df.count():,} linhas, {len(df.columns)} colunas")
df.printSchema()

## 3. Metadados de rastreabilidade

In [ ]:
df_bronze = (
    df
    .withColumn("_ingestion_ts", F.current_timestamp())
    .withColumn("_source_file", F.lit(ARQUIVO))
)

## 4. Escrita idempotente na bronze

In [ ]:
destino = f"{CATALOGO}.{SCHEMA_BRONZE}.{TABELA}"

(
    df_bronze.write
    .mode("overwrite")               # idempotente: roda de novo sobre o mesmo CSV, mesmo resultado
    .option("overwriteSchema", "true")
    .saveAsTable(destino)
)

print(f"{destino}: {spark.table(destino).count():,} linhas")

## 5. Verificação básica

Contagem de nulos/vazios por coluna — nenhum `display()`/`show()` de linhas inteiras se a tabela tiver
dado pessoal (`clientes`: CPF, nome, e-mail).

In [ ]:
_tabela_bronze = spark.table(destino)
display(
    _tabela_bronze.select(
        [F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), c)).alias(c)
         for c in _tabela_bronze.columns if not c.startswith("_")]
    )
)

## Checklist antes de comitar / dar push

- [ ] `TABELA`, `ARQUIVO`, `ENCODING` preenchidos corretamente (confira `data/raw/` e `CLAUDE.md`).
- [ ] Contagem de linhas bate com o esperado (ver `docs/achados_analise_geral.md`, seção Volumes).
- [ ] Nenhuma célula de output mostra linha de `clientes` (CPF/nome/e-mail).
- [ ] Rodei duas vezes e o resultado foi o mesmo (idempotente).
- [ ] Renomeei o arquivo pra `Landing_To_Bronze_<tabela>_<nome>.ipynb` antes de salvar/commitar.